# ReID-FM training on Kaggle

**GPU:** T4 x2 (or 1) · **Internet:** ON

1. Attach pool data (`POOL_INPUT_SLUG`) and optional splits / prior run.
2. Edit **only the CONFIG cell** — set `CONFIG_YAML` to a file under [`configs/kaggle/`](../configs/kaggle/README.md).
3. Run all cells.

**Default recipe:** `primary_sbs_r50_poolv2.yml` — ImageNet SBS R50-IBN on pool v2 (paper baseline).

In [ ]:
# =============================================================================
# CONFIG — one yaml knob (+ git + Kaggle inputs). See configs/kaggle/README.md
# =============================================================================
import os
import time

os.environ["REIDFM_SESSION_START"] = str(time.time())

# >>> CHANGE THIS ONLY for ablations <<<
CONFIG_YAML = "configs/kaggle/primary_sbs_r50_poolv2.yml"
# Examples:
#   configs/kaggle/primary_sbs_r50_poolv2.yml   # recommended SBS pool v2 (R0)
#   configs/kaggle/loss_study_B3.yml            # ArcFace + triplet ablation
#   configs/kaggle/fair_bot_r50_poolv2.yml      # appendix BoT baseline
#   configs/kaggle/smoke_sbs_r50.yml            # quick sanity run

GIT_REPO = "https://github.com/YOUR_ORG/Person-Re-ID-Foundational-model.git"
GIT_BRANCH = "main"
GIT_DEPTH = 1

RESTORE_FROM = ""       # e.g. "/kaggle/input/reidfm-train-s1" (previous session output)
FRESH_START = False
NUM_GPUS = 2
TIME_BUDGET_H = 11.0

POOL_INPUT_SLUG = "reidfm-pool-v2"
SPLITS_INPUT_SLUG = ""  # optional: pool_v2_train_ids.json + datasets/*.json

KAGGLE_INPUT = "/kaggle/input"
REPO_DIR = "/kaggle/working/reidfm-repo"
os.environ["REIDFM_DATA_ROOT"] = KAGGLE_INPUT
os.environ["REIDFM_OUTPUT_ROOT"] = "/kaggle/working/output"
os.environ["REIDFM_WEIGHTS_ROOT"] = "/kaggle/working/weights"
os.environ["REIDFM_TIME_BUDGET_H"] = str(TIME_BUDGET_H)

print("CONFIG_YAML:", CONFIG_YAML)

In [ ]:
import subprocess
from pathlib import Path

def sh(cmd: str) -> None:
    print("$", cmd)
    subprocess.check_call(cmd, shell=True)

repo = Path(REPO_DIR)
repo.parent.mkdir(parents=True, exist_ok=True)
if repo.is_dir() and (repo / ".git").is_dir():
    sh(f"cd {repo} && git fetch origin {GIT_BRANCH} --depth={GIT_DEPTH} && git checkout {GIT_BRANCH} && git pull --ff-only origin {GIT_BRANCH}")
else:
    if repo.exists():
        sh(f"rm -rf {repo}")
    sh(f"git clone --depth {GIT_DEPTH} --branch {GIT_BRANCH} {GIT_REPO} {repo}")

assert (repo / "scripts/train.py").is_file(), "git clone failed"
assert (repo / CONFIG_YAML).is_file(), f"missing {CONFIG_YAML} — pull latest main"

In [ ]:
import os, subprocess, sys
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "scripts/setup_fastreid.py", "--cython"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"])

In [ ]:
import os, shutil
from pathlib import Path

pool_root = Path(KAGGLE_INPUT) / POOL_INPUT_SLUG
assert pool_root.is_dir(), f"Attach dataset: {POOL_INPUT_SLUG}"
os.environ["REIDFM_DATA_ROOT"] = str(pool_root / "pool" if (pool_root / "pool").is_dir() else pool_root)

if SPLITS_INPUT_SLUG:
    src = Path(KAGGLE_INPUT) / SPLITS_INPUT_SLUG
    repo_splits = Path(REPO_DIR) / "splits"
    repo_splits.mkdir(parents=True, exist_ok=True)
    for name in ("pool_v2_train_ids.json", "pool_v2_val_ids.json"):
        if (src / name).is_file():
            shutil.copy2(src / name, repo_splits / name)
    dst = repo_splits / "datasets"
    dst.mkdir(parents=True, exist_ok=True)
    for j in list(src.glob("datasets/*.json")) + list(src.glob("*.json")):
        if j.name.startswith("pool_v2"):
            continue
        shutil.copy2(j, dst / j.name)
    os.environ["REIDFM_SPLITS_ROOT"] = str(dst)

In [ ]:
import os, sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
from reidfm.config import load_cfg
from reidfm.data.manifest import find_manifest
from reidfm.data.pool import load_pool

cfg = load_cfg(CONFIG_YAML)
pool = load_pool(cfg.REIDFM.POOL)
missing = []
for key in pool.sources:
    try:
        find_manifest(key)
    except FileNotFoundError:
        missing.append(key)
assert not missing, missing
print("OK:", CONFIG_YAML, "| init:", cfg.REIDFM.INIT_MODEL, "| out:", cfg.OUTPUT_DIR)

In [ ]:
import os, subprocess, sys
os.chdir(REPO_DIR)
args = [
    sys.executable, "scripts/train.py",
    "--config-file", CONFIG_YAML,
    "--model", cfg.REIDFM.INIT_MODEL,
    "--num-gpus", str(NUM_GPUS),
]
if RESTORE_FROM:
    args.extend(["--restore-from", RESTORE_FROM])
if RESTORE_FROM and not FRESH_START:
    args.append("--resume")
if FRESH_START:
    args.append("--fresh")
print(" ".join(args))
subprocess.check_call(args)

In [ ]:
from pathlib import Path
run_dir = Path("/kaggle/working/output") / cfg.OUTPUT_DIR
if (run_dir / "last_checkpoint").is_file():
    print((run_dir / "last_checkpoint").read_text())
for p in sorted(run_dir.glob("*"))[:15] if run_dir.is_dir() else []:
    print(p.name)
print("Resume: attach output as Kaggle dataset, set RESTORE_FROM to its path.")